In [ ]:
import polars as pl
from pathlib import Path

In [ ]:
"""
Lê os arquivos Parquet de bilhetagem revisados (entregues via SEI) e monta duas bases:
- Base de domingos: período completo (2023-2024), um arquivo por semana (sufixo "D").
- Base de 4 meses: abril, maio, setembro e outubro de 2023 e 2024, todos os dias
  (sufixos "S" = sábado e "U" = dia útil).
"""
# Raiz das fontes brutas (fora do repo). Se a unidade mudar, muda so esta linha.
RAIZ_SPTRANS = Path(r"C:\Users\9837292\Desktop\SSD\SPTrans")

pasta_domingos = str(RAIZ_SPTRANS / "SEI! 5010.2026-0008307-9")
pasta_4meses = str(RAIZ_SPTRANS / "SEI! 5010.2026-0011927-8")

# Convenção do repositório: outputs vão para ../outputs/<NN>/, onde NN é o prefixo
# da pasta do notebook que os gerou (este está em 01_criacao_de_bases/).
PASTA_SAIDA = Path("../outputs/01")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

caminho_domingos = str(PASTA_SAIDA / "Dados_Domingos_2023_2024.parquet")
caminho_4meses = str(PASTA_SAIDA / "Dados_4Meses_2023_2024.parquet")

In [ ]:
# Base de domingos: um arquivo por semana, período completo
arquivos_domingos = sorted(Path(pasta_domingos).glob("*.parquet"))
print(f"{len(arquivos_domingos)} arquivos de domingo encontrados.")

# scan_parquet + sink_parquet: streaming, não carrega tudo na memória de uma vez
pl.scan_parquet([str(f) for f in arquivos_domingos]).sink_parquet(caminho_domingos)

resumo = pl.scan_parquet(caminho_domingos).select(
    pl.len().alias("linhas"), pl.col("data").n_unique().alias("datas")
).collect()
print(f"Base de domingos: {resumo['linhas'][0]} linhas, {resumo['datas'][0]} datas distintas.")
print(f"Salvo em: {caminho_domingos}")

In [ ]:
# Base de 4 meses: todos os dias de abr/mai/set/out (2023 e 2024), deduplicando arquivos " (1)"
todos_arquivos_4meses = sorted(Path(pasta_4meses).glob("*.parquet"))

arquivos_por_nome = {}
for f in todos_arquivos_4meses:
    nome_normalizado = f.name.replace(" (1)", "")
    arquivos_por_nome.setdefault(nome_normalizado, f)  # mantém só a primeira ocorrência

arquivos_4meses = sorted(arquivos_por_nome.values())
print(f"{len(todos_arquivos_4meses)} arquivos encontrados, {len(arquivos_4meses)} após deduplicação.")

# scan_parquet + sink_parquet: streaming, não carrega tudo na memória de uma vez
pl.scan_parquet([str(f) for f in arquivos_4meses]).sink_parquet(caminho_4meses)

resumo = pl.scan_parquet(caminho_4meses).select(
    pl.len().alias("linhas"), pl.col("data").n_unique().alias("datas")
).collect()
print(f"Base de 4 meses: {resumo['linhas'][0]} linhas, {resumo['datas'][0]} datas distintas.")
print(f"Salvo em: {caminho_4meses}")